# Mean-Reversion Strategy — Backtester Example

This notebook demonstrates how to write a trading strategy using the Python interface
to the CMF C++ backtesting engine.

**Strategy logic:**  
- Track a rolling mid-price per instrument  
- When mid deviates more than `z_thresh` standard deviations from its `window`-tick mean,
  fade the move with a limit order  
- Hold one position per instrument at a time  

## 1. Setup

After building the C++ extension (`cmake --build build --target backtester_cpp`),
make sure the `python/` directory is on `PYTHONPATH`:

```bash
export PYTHONPATH="$PWD/python:$PYTHONPATH"
```
or run this cell:

In [ ]:
import sys
from pathlib import Path

# Add python/ to path so 'backtester' package is importable
repo_root = Path(".").resolve().parent
python_dir = repo_root / "python"
if str(python_dir) not in sys.path:
    sys.path.insert(0, str(python_dir))

print(f"Repo root : {repo_root}")
print(f"Data path : {repo_root / 'data'}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from backtester import Backtest, BacktestConfig, BacktestResult
from backtester_cpp import Strategy, Side, TimeInForce  # type: ignore

## 2. Strategy Definition

In [ ]:
class MeanReversionStrategy(Strategy):
    """Fade Z-score extremes using limit orders.

    Parameters
    ----------
    window    : rolling window length (number of mid-price ticks)
    z_thresh  : entry threshold in standard deviations
    """

    def __init__(self, window: int = 30, z_thresh: float = 2.0) -> None:
        super().__init__()
        self.window = window
        self.z_thresh = z_thresh
        self._history: dict[int, list[float]] = {}  # instrument_id -> mid-price history
        self._open_order: dict[int, int] = {}        # instrument_id -> order_id

    # ------------------------------------------------------------------
    # Callbacks
    # ------------------------------------------------------------------

    def on_book_update(
        self,
        instrument_id: int,
        timestamp_ns: int,
        bids: list[tuple[float, int]],
        asks: list[tuple[float, int]],
    ) -> None:
        if not bids or not asks:
            return

        mid = (bids[0][0] + asks[0][0]) / 2.0
        hist = self._history.setdefault(instrument_id, [])
        hist.append(mid)

        # Need at least window points
        if len(hist) < self.window:
            return

        # Keep only the last window ticks to bound memory
        window_prices = hist[-self.window :]
        mu = np.mean(window_prices)
        sigma = np.std(window_prices)

        if sigma < 1e-9:  # no dispersion — skip
            return

        z = (mid - mu) / sigma

        # Only open one order per instrument at a time
        if instrument_id in self._open_order:
            return

        best_bid_price = bids[0][0]
        best_ask_price = asks[0][0]

        if z > self.z_thresh:
            # Price is high relative to recent history → fade by selling
            oid = self.api.send_limit_order(
                instrument_id, Side.SELL, best_bid_price, 1, TimeInForce.GTC
            )
            self._open_order[instrument_id] = oid

        elif z < -self.z_thresh:
            # Price is low relative to recent history → fade by buying
            oid = self.api.send_limit_order(
                instrument_id, Side.BUY, best_ask_price, 1, TimeInForce.GTC
            )
            self._open_order[instrument_id] = oid

    def on_fill(
        self,
        order_id: int,
        instrument_id: int,
        timestamp_ns: int,
        price: float,
        size: int,
        side: Side,
    ) -> None:
        # Clear the open-order slot so we can trade again
        self._open_order.pop(instrument_id, None)

    def on_reject(
        self, order_id: int, instrument_id: int, reason: str
    ) -> None:
        self._open_order.pop(instrument_id, None)

## 3. Progress Callback

In [ ]:
def on_progress(info) -> None:
    ts_s = info.last_timestamp_ns / 1e9
    s = info.total_stats
    print(
        f"[progress] last_ts={ts_s:.0f}s  pnl={info.current_pnl:+.4f}  "
        f"orders: sent={s.sent} filled={s.filled} cancelled={s.cancelled} rejected={s.rejected}"
    )

## 4. Run Backtest

In [ ]:
DATA_PATH = repo_root / "data"

# Optional: restrict to a date window
# DATE_RANGE = ("2026-04-01", "2026-04-30")
DATE_RANGE = None

cfg = BacktestConfig()
cfg.book_levels       = 5          # top-5 levels passed to on_book_update
cfg.progress_interval = 30.0       # seconds between progress prints
cfg.progress_callback = on_progress

strategy = MeanReversionStrategy(window=30, z_thresh=2.0)

result: BacktestResult = Backtest(cfg).run(strategy, DATA_PATH, DATE_RANGE)
print(result)

## 5. Results

In [ ]:
print("=== Summary ===")
for k, v in result.summary.items():
    print(f"  {k:<18}: {v}")

In [ ]:
# PnL time series
result.plot_pnl()

In [ ]:
# Fill scatter
result.plot_fills()

In [ ]:
# First 10 fills
result.fills_df.head(10)

In [ ]:
# Order log status distribution
result.order_log_df["status"].value_counts()

In [ ]:
# Fills by instrument
result.fills_df.groupby("instrument_id")[["size", "realized_pnl"]].sum()